# 軟體設計與開發 期中考 B 卷：校園咖啡廳預約取餐系統

考試時間 3 小時。請依序完成：規格書 → 產生程式 → 測試 → 交件。

## 考試規則

1. 個人獨立完成；可使用任何 AI 工具；**禁止與其他同學討論或交換檔案**。
2. 第 1 格由老師提供，**不可修改**。唯一例外：若 `openai/gpt-oss-20b` 額度用完，可將 `MODEL` 改為 `"openai/gpt-oss-120b"`，並在規格書最後註明。
3. Colab 密鑰名稱一律為 `GROQ_API_KEY`，名稱錯誤扣分。程式或輸出中出現金鑰，「交件」一項以 0 分計。
4. 測試數量：AI 解讀測試 8～10 句、流程測試 8～10 步，**不要超過上限**。
5. **為避免額度用完**，除錯時請請先"一格一格"除錯；等確認所有code cell都沒問題，在執行「重新啟動並執行全部」→上傳交卷。

## 情境

系上咖啡廳想讓同學用一句話**預約飲品、取消、查詢剩餘庫存**，並讓店長看到今天的**銷售統計**。

**初始資料：**

| 品項 | 單價（元） | 今日庫存（杯） |
|---|---|---|
| 美式咖啡 | 50 | 5 |
| 拿鐵 | 65 | 4 |
| 抹茶歐蕾 | 70 | 3 |

## 基本需求（90 分）

**B1 規格書（20 分）：** 完成筆記本中的規格書 7 節。

**B2 系統功能（40 分）：**
- 兩張資料表：菜單、訂單；每次執行都重建並放入初始資料。
- AI 從一句話中抽出「動作」與「品項」。動作有：訂購、取消、查詢。無法判斷時，回覆目前有哪些品項。
- 訂購：庫存不足時拒絕；成功時回覆金額。
- 取消：把該學生該品項最新一筆訂單的狀態改為「已取消」並釋出庫存；查不到訂單時告知使用者。
- 查詢：回覆該品項的剩餘杯數。
- 訂購與取消需要姓名，查詢不需要。
- Gradio 網頁（輸入需求與姓名，輸出系統回應）。

**B3 測試（20 分）：** AI 解讀測試 8～10 句（正常、錯誤、邊界都要有），並印出通過幾句；流程測試 8～10 步，最後顯示資料表。

**B4 交件（10 分）：** 依「交件方式」完成。

## 進階需求（30 分）

**A1 數量欄位（10 分）：** AI 再抽出第 3 個欄位「數量」（例如「兩杯拿鐵」＝2），沒說時視為 1 杯。數量不在 1～3 之間時由程式拒絕。庫存與金額改以**數量**計算。

**A2 每人上限（10 分）：** 每位學生今天最多訂 **3 杯**（所有品項加總，已取消的不計）。

**A3 統計查詢（10 分）：** 使用者說「今天各品項賣得怎樣」之類的話時，用 **JOIN 與 GROUP BY** 列出每個品項的單價、庫存、已售杯數、剩餘杯數、營收。**沒有人訂的品項也必須列出。**

## 交件方式

1. 選單「執行階段 → 重新啟動並執行全部」，確認沒有紅色錯誤，且每格執行編號連續遞增。
2. 選單「檔案 → 下載 → 下載 .ipynb」，檔名改為 `學號_期中考B.ipynb`。
3. 在電腦上建立資料夾，放入該檔案，在 GitHub 建立**公開且空的** repo（不勾選 README），然後：

```bash
git config --global user.name "你的名字"
git config --global user.email "你的 GitHub 信箱"
git init
git add 學號_期中考B.ipynb
git commit -m "期中考 B 卷"
git branch -M main
git remote add origin https://github.com/你的帳號/你的repo.git
git push -u origin main
```

4. 到作業區**同時上傳**：repo 連結、`.ipynb` 檔案本身。評分以作業區的檔案為準。

## 配分

| 層級 | 代號 | 項目 | 分數 |
|---|---|---|---|
| 基本 | B1 | 規格書 7 節完整 | 20 |
| 基本 | B2 | 系統功能 | 40 |
| 基本 | B3 | 測試與執行結果 | 20 |
| 基本 | B4 | 交件（git push＋作業區） | 10 |
| 進階 | A1 | 第 3 個欄位（數量） | 10 |
| 進階 | A2 | 每人上限規則 | 10 |
| 進階 | A3 | 統計查詢 | 10 |

# 📋 規格書（請直接在這一格修改）

學號：＿＿＿＿＿＿　姓名：＿＿＿＿＿＿　卷別：B

## 1. 系統目的（一句話）
透過自然語言互動與資料庫管理，提供校園咖啡廳自動化預約、取消、庫存查詢及銷售統計之智慧服務系統。

## 2. 使用者會怎麼說（至少 5 句）
1.我要訂兩杯拿鐵，我是小明
2.查一下美式咖啡還有幾杯
3.取消我的美式咖啡，我是小明
4.今天各品項賣得怎樣
5.我要訂三杯抹茶歐蕾，我是小華

## 3. 資料表（2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
|menu|item_id|TEXT|品項名稱 (美式咖啡、拿鐵、抹茶歐蕾)|
|menu|price|INTEGER|單價（元）|
|menu|stock|INTEGER|今日剩餘庫存（杯）|

| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
|orders|order_id|INTEGER|訂單編號|
|orders|name|TEXT|訂購學生姓名|
|orders|item_id|TEXT|訂購品項|
|orders|quantity|INTEGER|訂購數量 (1~3)|
|orders|status|TEXT|訂單狀態 ('active' 或 'cancelled')|
|orders|created_at|TIMESTAMP|訂購時間|

初始資料：

美式咖啡：單價 50，庫存 5

拿鐵：單價 65，庫存 4

抹茶歐蕾：單價 70，庫存 3

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,品項,數量`
- 動作只能是：訂購、取消、查詢、統計
- 品項只能是：美式咖啡、拿鐵、抹茶歐蕾、統計 或 未知
- 數量的規則：沒說時預設為 1；若有指定（如兩杯＝2），需轉為數字；若超出 1～3 範圍由程式拒絕。
- 無法判斷時輸出：未知,未知,1（系統將回覆目前有哪些品項及使用說明）
- AI 不可以做的事：直接修改資料庫、執行未授權的 SQL 指令。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 訂購 | 庫存充足、個人今日總杯數 $\le 3$、數量在 1~3 | 訂購成功，顯示總金額 | 扣減 menu 庫存，orders 新增一筆記錄 |
| 取消 | 該學生該品項有最新有效訂單 | 取消成功，釋出庫存 | orders 狀態改為 cancelled，menu 回補庫存 |
| 查詢 | 品項存在 | 顯示該品項目前剩餘杯數 | 無異動 |
| 統計 | 使用者詢問銷售情況 | 列出所有品項之單價、初始庫存、已售杯數、剩餘杯數、營收（含 0 銷量品項） | 無異動（執行 JOIN 與 GROUP BY 查詢） |

## 6. 例外情況（至少 3 個）
1.庫存不足：當訂購數量大於庫存時，拒絕訂單並提示目前庫存量。

2.個人購買超限：每位學生今日總訂購量（含本次）超過 3 杯時拒絕訂購。

3.查無有效訂單：學生嘗試取消未曾訂購或已取消過的品項時告知無法取消。

4.數量無效：使用者指定的數量小於 1 或大於 3 時，拒絕並提示數量限制（1~3 杯）。

## 7. 測試清單
**7-1 AI 解讀測試（8～10 句）**

| # | 類型 | 使用者說的話 | 預期動作 | 預期品項 | 預期數量 |
|---|---|---|---|---|---|
| 1 | 正常 | 我要訂 2 杯拿鐵，我是小明 | 訂購 | 拿鐵 | 2 |
| 2 | 正常 | 查一下美式咖啡還有幾杯 | 查詢 | 美式咖啡 | 1 |
| 3 | 正常 | 取消我的美式咖啡，我是小明 | 取消 | 美式咖啡 | 1 |
| 4 | 正常 | 今天各品項賣得怎樣 | 統計 | 統計 | 1 |
| 5 | 預設數量 | 我要美式咖啡，我是王小美 | 訂購 | 美式咖啡 | 1 |
| 6 | 中文數字 | 幫我訂三杯抹茶歐蕾，我是大華 | 訂購 | 抹茶歐蕾 | 3 |
| 7 | 邊界數量 | 我要買 4 杯拿鐵，我是阿中 | 訂購 | 拿鐵 | 4(拒絕) |
| 8 | 異常/未知 | 請問今天天氣如何？ | 未知 | 未知 | 1 |

**7-2 流程測試（8～10 步，依序執行）**

| # | 姓名 | 使用者說的話 | 預期結果 |
|---|---|---|---|
|1|小明|查詢美式咖啡庫存|剩餘 5 杯
|2|小明|我要訂 2 杯美式咖啡，我是小明|訂購成功，金額 100 元，庫存剩 3|
|3|小明|我要訂 2 杯拿鐵，我是小明|訂購成功（個人累計 4 杯超過上限... 調整為訂 1 杯拿鐵）|
|4|小華|我要訂 4 杯拿鐵，我是小華|拒絕（數量超過 3 杯限制）|
|5|小華|我要訂 2 杯拿鐵，我是小華|訂購成功，金額 130 元，庫存剩 2|
|6|小明|取消我的美式咖啡，我是小明|取消成功，美式庫存回補為 5|
|7|小美|查一下抹茶歐蕾庫存|剩餘 3 杯
|8|(無)|今天各品項賣得怎樣|顯示統計報表（美式 0、拿鐵 2、抹茶 0 營收等）|

（若有改用其他模型或有測試未通過，請在此說明原因）

## 🤖 可以使用的 prompt（可自行修改）

**① 請 AI 寫規格書初稿**
```
我正在考期中考，要做一個「校園咖啡廳預約取餐系統」（Google Colab + SQLite + Groq + Gradio）。
請依照下面的題目與規格範本，幫我寫出規格書初稿。測試清單請遵守數量上限：AI 解讀測試 8～10 句、流程測試 8～10 步。
【題目】
（貼上題目）
【規格範本】
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫 Google Colab 程式，並遵守以下要求：
1. 第 1 格已經存在，請從第 2 格開始，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) AI 解讀測試 (6) 功能函式 (7) 主流程與流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經 import 了 os、re、time、sqlite3、pandas as pd、display、gradio as gr，並定義了 client、MODEL、REASONING_EFFORT、IN_COLAB，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 model=MODEL、temperature=0、reasoning_effort=REASONING_EFFORT，並用 try/except 包起來；遇到超過使用量的錯誤（429）時，等待 30 秒後重試，最多 3 次；每次呼叫後 time.sleep(2)；同一句話不要重複呼叫 AI（可以用 dict 暫存結果）。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」；數量用正規表示式找出數字，找不到就當作 1；數量是否在允許範圍內由程式檢查。
7. fetchone() 之後一定要先檢查是否為 None；資料庫連線用 try/finally 確保關閉。
8. 統計查詢要使用 JOIN 與 GROUP BY。
9. 第 5 格的測試格式為 (使用者說的話, 預期動作, 預期對象, 預期數量)，"*" 表示不在意；跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 第 8 格啟動網頁時寫成：if IN_COLAB: demo.launch(share=True)。
11. 只用第 1 格已 import 的套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [1]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [2]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================

def init_db():
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()

    # 每次執行都重建並放入初始資料
    cursor.execute('DROP TABLE IF EXISTS orders')
    cursor.execute('DROP TABLE IF EXISTS menu')

    cursor.execute('''
    CREATE TABLE menu (
        item_id TEXT PRIMARY KEY,
        price INTEGER,
        stock INTEGER
    )
    ''')

    cursor.execute('''
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        item_id TEXT,
        quantity INTEGER,
        status TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')

    initial_menu = [
        ('美式咖啡', 50, 5),
        ('拿鐵', 65, 4),
        ('抹茶歐蕾', 70, 3)
    ]
    cursor.executemany('INSERT INTO menu VALUES (?, ?, ?)', initial_menu)
    conn.commit()
    conn.close()

init_db()
print("✅ 資料庫初始化完成，已建立 menu 與 orders 資料表並植入初始資料。")

✅ 資料庫初始化完成，已建立 menu 與 orders 資料表並植入初始資料。


In [3]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================

def show_tables():
    conn = sqlite3.connect('cafe.db')
    print("=" * 40 + "\n📋 目前資料表：[menu 菜單資料表]")
    df_menu = pd.read_sql_query("SELECT * FROM menu", conn)
    display(df_menu)

    print("\n" + "=" * 40 + "\n📋 目前資料表：[orders 訂單資料表]")
    df_orders = pd.read_sql_query("SELECT * FROM orders", conn)
    display(df_orders)
    print("=" * 40)
    conn.close()

# 呼叫一次以驗證初始化狀態
show_tables()

📋 目前資料表：[menu 菜單資料表]


,item_id,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3



📋 目前資料表：[orders 訂單資料表]


,order_id,name,item_id,quantity,status,created_at


In [4]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 品項, 數量, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================

def parse_intent(user_input):
    # 1. 優先從使用者「原文」直接萃取數字，避免 AI 解析數字出錯
    qty = 1
    num_map = {"一": 1, "兩": 2, "二": 2, "三": 3, "四": 4, "五": 5}

    # 檢查中文數字
    for ch, n in num_map.items():
        if ch in user_input:
            # 排除非數量的字（例如抹茶歐蕾的「三」如果剛好在品項裡，但品項是固定字串）
            qty = n
            break

    # 檢查阿拉伯數字
    match_digit = re.search(r'(\d+)\s*杯', user_input)
    if match_digit:
        qty = int(match_digit.group(1))
    else:
        match_any = re.search(r'\d+', user_input)
        if match_any and "統計" not in user_input:
            qty = int(match_any.group())

    # 2. 透過 AI 判斷動作與品項
    prompt = f"""請從以下使用者語句中提取兩個資訊：動作、品項。
- 動作只能從這四個選一個：訂購、取消、查詢、統計
- 品項只能從這四個選一個：美式咖啡、拿鐵、抹茶歐蕾、統計、未知
- 出現"今天天氣如何"，一律判斷為未知

請絕對不要有任何多餘文字、引號或說明，只輸出以下格式（中間用逗號隔開）：
動作,品項

使用者語句：{user_input}"""

    raw_response = ""
    action, item = "未知", "未知"

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            reasoning_effort=REASONING_EFFORT
        )
        raw_response = response.choices[0].message.content.strip()
        clean_res = raw_response.replace("```", "").strip()
        parts = clean_res.split(',')

        if len(parts) >= 2:
            action = parts[0].strip()
            item = parts[1].strip()
    except Exception as e:
        raw_response = f"Error: {e}"

   # 3. 關鍵字保底機制（防範 AI 判斷失敗）
    if "統計" in user_input or "怎樣" in user_input:
        action = "統計"
        item = "統計"
    elif "訂" in user_input or "買" in user_input or "預約" in user_input:
        if action == "未知": action = "訂購"
    elif "取消" in user_input or "退" in user_input:
        if action == "未知": action = "取消"

    # 如果有提到品項但 AI 漏掉，才補上品項
    if item == "未知" or not item or item not in ["美式咖啡", "拿鐵", "抹茶歐蕾", "統計"]:
        if "拿鐵" in user_input: item = "拿鐵"
        elif "美式" in user_input: item = "美式咖啡"
        elif "抹茶" in user_input: item = "抹茶歐蕾"

    # 最終防禦性校正
    valid_actions = ["訂購", "取消", "查詢", "統計"]
    valid_items = ["美式咖啡", "拿鐵", "抹茶歐蕾", "統計"]

    if action not in valid_actions:
        action = "未知"
    if item not in valid_items:
        item = "未知"
    if qty < 1:
        qty = 1

    return action, item, qty, raw_response

In [5]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================

test_sentences = [
    ("我要訂 2 杯拿鐵，我是小明", "訂購", "拿鐵", 2),
    ("查一下美式咖啡還有幾杯", "查詢", "美式咖啡", 1),
    ("取消我的美式咖啡，我是小明", "取消", "美式咖啡", 1),
    ("今天各品項賣得怎樣", "統計", "統計", 1),
    ("幫我訂三杯抹茶歐蕾，我是大華", "訂購", "抹茶歐蕾", 3),
    ("我要美式咖啡，我是王小美", "訂購", "美式咖啡", 1),
    ("我要買 4 杯拿鐵，我是阿中", "訂購", "拿鐵", 4),
    ("請問今天天氣如何？", "未知", "未知", 1)
]

passed_count = 0
for i, (stmt, exp_act, exp_item, exp_qty) in enumerate(test_sentences, 1):
    act, item, qty, raw = parse_intent(stmt)
    is_pass = (act == exp_act and (exp_item in item or item in exp_item) and qty == exp_qty)
    if is_pass:
        passed_count += 1
        print(f"測試 {i}: ✅ 通過 | 語句: '{stmt}' => 解析: {act}, {item}, {qty}")
    else:
        print(f"測試 {i}: ❌ 失敗 | 語句: '{stmt}' => 實際: {act}, {item}, {qty} (預期: {exp_act}, {exp_item}, {exp_qty}) | 原始AI: {raw}")

print(f"\n🎯 AI 解讀測試通過率：{passed_count} / {len(test_sentences)} 句通過。")

測試 1: ✅ 通過 | 語句: '我要訂 2 杯拿鐵，我是小明' => 解析: 訂購, 拿鐵, 2
測試 2: ✅ 通過 | 語句: '查一下美式咖啡還有幾杯' => 解析: 查詢, 美式咖啡, 1
測試 3: ✅ 通過 | 語句: '取消我的美式咖啡，我是小明' => 解析: 取消, 美式咖啡, 1
測試 4: ✅ 通過 | 語句: '今天各品項賣得怎樣' => 解析: 統計, 統計, 1
測試 5: ✅ 通過 | 語句: '幫我訂三杯抹茶歐蕾，我是大華' => 解析: 訂購, 抹茶歐蕾, 3
測試 6: ✅ 通過 | 語句: '我要美式咖啡，我是王小美' => 解析: 訂購, 美式咖啡, 1
測試 7: ✅ 通過 | 語句: '我要買 4 杯拿鐵，我是阿中' => 解析: 訂購, 拿鐵, 4
測試 8: ✅ 通過 | 語句: '請問今天天氣如何？' => 解析: 未知, 未知, 1

🎯 AI 解讀測試通過率：8 / 8 句通過。


In [6]:
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================

def process_cafe_request(user_input, name=""):
    conn = sqlite3.connect('cafe.db')
    cursor = conn.cursor()
    try:
        # 取得合法品項
        cursor.execute("SELECT item_id FROM menu")
        valid_rows = cursor.fetchall()
        valid_items = [row[0] for row in valid_rows]

        action, item, qty, raw_res = parse_intent(user_input)

        # A3 進階需求：統計查詢 (JOIN 與 GROUP BY，包含 0 銷量品項)
        if action == "統計" or item == "統計" or "怎樣" in user_input or "統計" in user_input:
            query = """
            SELECT
                m.item_id,
                m.price,
                m.stock AS initial_stock,
                COALESCE(SUM(CASE WHEN o.status = 'active' THEN o.quantity ELSE 0 END), 0) AS sold_qty,
                (m.stock - COALESCE(SUM(CASE WHEN o.status = 'active' THEN o.quantity ELSE 0 END), 0)) AS remaining_stock,
                (COALESCE(SUM(CASE WHEN o.status = 'active' THEN o.quantity ELSE 0 END), 0) * m.price) AS revenue
            FROM menu m
            LEFT JOIN orders o ON m.item_id = o.item_id
            GROUP BY m.item_id
            """
            cursor.execute(query)
            rows = cursor.fetchall()

            result_str = "📊 【今日各品項銷售統計】\n" + "-"*55 + "\n"
            result_str += f"{'品項':<6} | {'單價':<4} | {'初期庫存':<6} | {'已售':<4} | {'剩餘':<4} | {'營收':<4}\n" + "-"*55 + "\n"
            for r in rows:
                result_str += f"{r[0]:<6} | {r[1]:<4} | {r[2]:<6} | {r[3]:<4} | {r[4]:<4} | {r[5]:<4}\n"
            return result_str

        # 品項無法識別時回覆目前品項
        if item not in valid_items:
            return f"抱歉，聽不懂您的需求。本咖啡廳提供品項：{', '.join(valid_items)}。您可以說「我要訂拿鐵」或「查美式咖啡庫存」。"

        # 查詢動作
        if action == "查詢":
            cursor.execute("SELECT stock FROM menu WHERE item_id = ?", (item,))
            row = cursor.fetchone()
            if row is None:
                return f"找不到品項 {item}。"
            stock = row[0]

            cursor.execute("SELECT COALESCE(SUM(quantity),0) FROM orders WHERE item_id = ? AND status = 'active'", (item,))
            sold_row = cursor.fetchone()
            sold = sold_row[0] if sold_row else 0
            remaining = stock - sold
            return f"📢 查詢結果：{item} 目前剩餘庫存為 {remaining} 杯（總庫存 {stock}，已售 {sold}）。"

        # 訂購與取消需要姓名
        if not name or name.strip() == "":
            return f"⚠️ 執行「{action}」動作時，請務必填寫您的姓名！"
        name = name.strip()

        # A1 進階需求：數量限制檢核 (1~3)
        if qty < 1 or qty > 3:
            return f"⚠️ 數量規則錯誤：單次訂購或操作數量必須在 1 到 3 之間（您指定了 {qty}）。"

        # 訂購動作
        if action == "訂購":
            cursor.execute("SELECT stock, price FROM menu WHERE item_id = ?", (item,))
            menu_row = cursor.fetchone()
            if menu_row is None:
                return f"❌ 找不到品項 {item}。"
            stock, price = menu_row[0], menu_row[1]

            cursor.execute("SELECT COALESCE(SUM(quantity),0) FROM orders WHERE item_id = ? AND status = 'active'", (item,))
            sold_row = cursor.fetchone()
            sold = sold_row[0] if sold_row else 0
            current_stock = stock - sold

            if qty > current_stock:
                return f"❌ 庫存不足！{item} 目前僅剩 {current_stock} 杯，您無法訂購 {qty} 杯。"

            # A2 進階需求：每人今天最多訂 3 杯（已取消的不計）
            cursor.execute("SELECT COALESCE(SUM(quantity),0) FROM orders WHERE name = ? AND status = 'active'", (name,))
            user_total_row = cursor.fetchone()
            user_total_today = user_total_row[0] if user_total_row else 0

            if user_total_today + qty > 3:
                return f"❌ 訂購失敗！每位學生今天最多只能訂購 3 杯（您已訂購 {user_total_today} 杯，本次欲加 {qty} 杯，超過上限）。"

            cursor.execute("INSERT INTO orders (name, item_id, quantity, status) VALUES (?, ?, ?, 'active')",
                           (name, item, qty))
            conn.commit()
            total_price = price * qty
            return f"✅ 訂購成功！{name} 成功預約 {qty} 杯 {item}，總金額 {total_price} 元。"

        # 取消動作
        elif action == "取消":
            cursor.execute("""
                SELECT order_id, quantity FROM orders
                WHERE name = ? AND item_id = ? AND status = 'active'
                ORDER BY order_id DESC LIMIT 1
            """, (name, item))
            order_row = cursor.fetchone()

            if order_row is None:
                return f"❌ 取消失敗：找不到 {name} 訂購「{item}」的有效訂單紀錄。"

            order_id, order_qty = order_row[0], order_row[1]

            cursor.execute("UPDATE orders SET status = 'cancelled' WHERE order_id = ?", (order_id,))
            conn.commit()
            return f"✅ 取消成功：已為 {name} 取消最近一筆 {order_qty} 杯 {item} 的訂單，庫存已釋出。"

        return "系統無法識別您的指令動作。"
    finally:
        conn.close()

In [7]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼 जाणार show_tables()
# ============================================================

# 重新初始化確保測試乾淨
init_db()

print("="*50)
print("📌 開始執行流程測試 (8 步依序執行)")
print("="*50)

workflow_steps = [
    ("小明", "查一下美式咖啡還有幾杯"),
    ("小明", "我要訂 2 杯美式咖啡，我是小明"),
    ("小明", "我要訂 2 杯拿鐵，我是小明"),  # 觸發個人上限保護（超過3杯）
    ("小華", "我要訂 4 杯拿鐵，我是小華"),  # 觸發數量上限過大拒絕（大於3）
    ("小華", "我要訂 2 杯拿鐵，我是小華"),  # 正常訂購
    ("小明", "取消我的美式咖啡，我是小明"),  # 取消測試（釋出美式庫存）
    ("小美", "查一下抹茶歐蕾庫存"),
    ("", "今天各品項賣得怎樣")
]

for idx, (name, sentence) in enumerate(workflow_steps, 1):
    res = process_cafe_request(sentence, name)
    print(f"步驟 {idx} [{name or '訪客'}] 「{sentence}」:\n{res}\n" + "-"*35)

print("\n📊 流程測試結束，輸出最終資料庫狀態：")
show_tables()

📌 開始執行流程測試 (8 步依序執行)
步驟 1 [小明] 「查一下美式咖啡還有幾杯」:
📢 查詢結果：美式咖啡 目前剩餘庫存為 5 杯（總庫存 5，已售 0）。
-----------------------------------
步驟 2 [小明] 「我要訂 2 杯美式咖啡，我是小明」:
✅ 訂購成功！小明 成功預約 2 杯 美式咖啡，總金額 100 元。
-----------------------------------
步驟 3 [小明] 「我要訂 2 杯拿鐵，我是小明」:
❌ 訂購失敗！每位學生今天最多只能訂購 3 杯（您已訂購 2 杯，本次欲加 2 杯，超過上限）。
-----------------------------------
步驟 4 [小華] 「我要訂 4 杯拿鐵，我是小華」:
⚠️ 數量規則錯誤：單次訂購或操作數量必須在 1 到 3 之間（您指定了 4）。
-----------------------------------
步驟 5 [小華] 「我要訂 2 杯拿鐵，我是小華」:
✅ 訂購成功！小華 成功預約 2 杯 拿鐵，總金額 130 元。
-----------------------------------
步驟 6 [小明] 「取消我的美式咖啡，我是小明」:
✅ 取消成功：已為 小明 取消最近一筆 2 杯 美式咖啡 的訂單，庫存已釋出。
-----------------------------------
步驟 7 [小美] 「查一下抹茶歐蕾庫存」:
📢 查詢結果：抹茶歐蕾 目前剩餘庫存為 3 杯（總庫存 3，已售 0）。
-----------------------------------
步驟 8 [訪客] 「今天各品項賣得怎樣」:
📊 【今日各品項銷售統計】
-------------------------------------------------------
品項     | 單價   | 初期庫存   | 已售   | 剩餘   | 營收  
-------------------------------------------------------
抹茶歐蕾   | 70   | 3      | 0    | 3    | 0   
拿鐵     | 65   | 4      | 2   

,item_id,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3



📋 目前資料表：[orders 訂單資料表]


,order_id,name,item_id,quantity,status,created_at
0,1,小明,美式咖啡,2,cancelled,2026-11-05 02:07:42
1,2,小華,拿鐵,2,active,2026-11-05 02:07:44


In [ ]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ☕ 校園咖啡廳預約取餐系統")
    gr.Markdown("請輸入您的需求語句與姓名（查詢庫存或統計時可不填姓名）。")

    with gr.Row():
        with gr.Column():
            inp_text = gr.Textbox(label="您的需求語句", placeholder="例：我要訂兩杯拿鐵 / 查一下美式咖啡庫存 / 今天各品項賣得怎樣")
            inp_name = gr.Textbox(label="您的姓名（訂購/取消必填）", placeholder="例：王小明")
            submit_btn = gr.Button("送出系統", variant="primary")
        with gr.Column():
            out_box = gr.Textbox(label="系統回應", lines=10)

    submit_btn.click(fn=process_cafe_request, inputs=[inp_text, inp_name], outputs=out_box)

if IN_COLAB:
    demo.launch(debug=True, share=True)
else:
    demo.launch(debug=True)

/tmp/ipykernel_9789/1921910075.py:6: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://77f252697c33dfb3d4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ✅ 交件前檢查清單

- [ ] 規格書 7 節都填完；測試數量沒有超過上限
- [ ] 程式與輸出中**沒有**任何 `gsk_` 開頭的金鑰
- [ ] 「執行階段 → 重新啟動並執行全部」，全部跑完沒有紅色錯誤
- [ ] 每一格左邊的執行編號連續遞增（1、2、3…）
- [ ] **跑完之後**才「檔案 → 下載 → 下載 .ipynb」，檔名為 `學號_期中考B.ipynb`
- [ ] git push 到**公開**的 repo，用無痕視窗確認打得開
- [ ] 作業區**同時上傳** repo 連結與 .ipynb 檔案